<a href="https://colab.research.google.com/github/Nitish-Malik/Pytorch_project/blob/main/Optuna_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install Optuna

In [12]:
from os import name
# Importing Necessary Libraries

import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Note: Sklearn bulitin dataset is a regression dataset hence will also import the actual dataset
# Pima Indian Diabetes datasets.

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetespedigreeFunction',
           'Age', 'Outcome']

# Load the datasets
import pandas as pd
import numpy as np

df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetespedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [13]:
# Replace zero values with NaN
cols_with_missing_val = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_val] = df[cols_with_missing_val].replace(0, np.nan)


#Impute the missing values with the mean of the respective col
df.fillna(df.mean(), inplace=True)
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetespedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [14]:
from pandas.core.tools.datetimes import Scalar
# Spliting the outcomes
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training set Shape: {X_train.shape}")
print(f"Test set Shape: {X_test.shape}")

Training set Shape: (537, 8)
Test set Shape: (231, 8)


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the Objective Function
def objective(trail):
  n_estimators = trail.suggest_int('n_estimators', 50, 200)
  max_depth = trail.suggest_int('max_depth', 3, 20)

  # Create RandomForestClassifier with suggestd Hyperparameters
  model = RandomForestClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      random_state=42
  )

  score = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()

  return score

In [16]:
# Create a study object and Optimize the Objective function
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-11-03 06:07:12,145] A new study created in memory with name: no-name-d52ccf14-af5a-40db-b273-8df99b51daab
[I 2026-11-03 06:07:12,975] Trial 0 finished with value: 0.7765363128491619 and parameters: {'n_estimators': 148, 'max_depth': 7}. Best is trial 0 with value: 0.7765363128491619.
[I 2026-11-03 06:07:13,523] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 96, 'max_depth': 19}. Best is trial 0 with value: 0.7765363128491619.
[I 2026-11-03 06:07:13,830] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 55, 'max_depth': 8}. Best is trial 0 with value: 0.7765363128491619.
[I 2026-11-03 06:07:14,718] Trial 3 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 159, 'max_depth': 19}. Best is trial 0 with value: 0.7765363128491619.
[I 2026-11-03 06:07:15,168] Trial 4 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 79, 'max_depth': 8}. Best is trial 0 with value: 0.7765363128

In [17]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 56, 'max_depth': 7}


In [18]:
# Creating model using the best hyperparameters
from sklearn.metrics import  accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy with best parameter : {test_accuracy}")


Test Accuracy with best parameter : 0.7402597402597403


In [19]:
# Using Random Sampler this time
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.RandomSampler())
study.optimize(objective, n_trials=50)

[I 2026-11-03 06:07:45,266] A new study created in memory with name: no-name-f3c574c1-a712-47e7-8dab-1a51cec0aecd
[I 2026-11-03 06:07:46,009] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 119, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-03 06:07:47,067] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 175, 'max_depth': 8}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-03 06:07:47,542] Trial 2 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 67, 'max_depth': 18}. Best is trial 2 with value: 0.7783985102420857.
[I 2026-11-03 06:07:48,214] Trial 3 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 111, 'max_depth': 20}. Best is trial 2 with value: 0.7783985102420857.
[I 2026-11-03 06:07:49,305] Trial 4 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 194, 'max_depth': 6}. Best is trial 2 with value: 0.77839851

In [20]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

# Creating model using the best hyperparameters
from sklearn.metrics import  accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy with best parameter : {test_accuracy}")

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 57, 'max_depth': 7}
Test Accuracy with best parameter : 0.7402597402597403


In [21]:
# Using the GridSampler

search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-11-03 06:08:21,264] A new study created in memory with name: no-name-6fac9563-2367-419b-8026-22e56c0699a8
[I 2026-11-03 06:08:21,790] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-03 06:08:22,677] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-03 06:08:22,967] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-11-03 06:08:23,525] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-11-03 06:08:24,064] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [22]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

# Creating model using the best hyperparameters
from sklearn.metrics import  accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy with best parameter : {test_accuracy}")

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}
Test Accuracy with best parameter : 0.7359307359307359


In [23]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [24]:
# Optimization
plot_optimization_history(study).show()

In [25]:
# 2. Prallel coordinate plot
plot_parallel_coordinate(study).show()


In [26]:
# 3. Sclice Plot
plot_slice(study).show()

In [27]:
plot_contour(study).show()

In [28]:
plot_param_importances(study).show()

## Optimizing Multiple ML models

- Optuna allows us to select the best model with their best hyperparamters.

In [29]:
# Importing Libraries
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC

In [39]:
# Define the Objective function for Optuna
def objective(trial):
  classifer_name = trial.suggest_categorical('classifer', ['SVM', 'RandomForest', 'GradientBoosting'])

  if classifer_name == 'SVM':
    #SVM Hyperparameters
    c = trial.suggest_float('c', 0.1, 100, log=True)
    kernal = trial.suggest_categorical('kernal', ['linear', 'rbf', 'poly', 'sigmoid'])
    gamma = trial.suggest_categorical('gamma',['scale', 'auto'])

    model = SVC(C=c, kernel=kernal, gamma=gamma, random_state=42)

  elif classifer_name == 'RandomForest':
    # Random Forest Hyperparameters
    model = RandomForestClassifier(
        n_estimators = trial.suggest_int('n_estimators',50, 300),
        max_depth = trial.suggest_int('max_depth', 3,20),
        min_samples_split = trial.suggest_int('min_samples_split',2,10),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10),
        bootstrap= trial.suggest_categorical('bootstrap',[True, False]),
        random_state = 42)

  elif classifer_name == 'GradientBoosting':
    # Gradient Boosting hyperparameters
    model = GradientBoostingClassifier(
        n_estimators = trial.suggest_int('n_estimators', 50, 300),
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        max_depth = trial.suggest_int('max_depth', 3, 20),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10),
        random_state = 42)


  # Perfomance Cross-validation and return the mean accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
  return score

In [40]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials = 100)

[I 2026-11-03 06:14:00,152] A new study created in memory with name: no-name-cf73159e-088d-4fef-9950-1853bcc6e9ab
[I 2026-11-03 06:14:00,194] Trial 0 finished with value: 0.756052141527002 and parameters: {'classifer': 'SVM', 'c': 4.895865514770521, 'kernal': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.756052141527002.
[I 2026-11-03 06:14:00,230] Trial 1 finished with value: 0.7728119180633147 and parameters: {'classifer': 'SVM', 'c': 0.18488022587684433, 'kernal': 'sigmoid', 'gamma': 'scale'}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-03 06:14:03,316] Trial 2 finished with value: 0.7616387337057727 and parameters: {'classifer': 'GradientBoosting', 'n_estimators': 298, 'learning_rate': 0.010999315510515657, 'max_depth': 12, 'min_samples_split': 3, 'min_samples_leaf': 8}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-03 06:14:04,778] Trial 3 finished with value: 0.7765363128491621 and parameters: {'classifer': 'GradientBoosting', 'n_estimators':

In [41]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifer': 'SVM', 'c': 0.13265353775851677, 'kernal': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [44]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_bootstrap,params_c,params_classifer,params_gamma,params_kernal,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.756052,2026-11-03 06:14:00.154357,2026-11-03 06:14:00.194330,0 days 00:00:00.039973,NaN,4.895866,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.772812,2026-11-03 06:14:00.195399,2026-11-03 06:14:00.230810,0 days 00:00:00.035411,NaN,0.184880,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.761639,2026-11-03 06:14:00.231759,2026-11-03 06:14:03.316321,0 days 00:00:03.084562,NaN,NaN,GradientBoosting,NaN,NaN,0.010999,12.0,8.0,3.0,298.0,NaN,COMPLETE
3,3,0.776536,2026-11-03 06:14:03.317288,2026-11-03 06:14:04.778673,0 days 00:00:01.461385,NaN,NaN,GradientBoosting,NaN,NaN,0.021774,4.0,8.0,4.0,254.0,NaN,COMPLETE
4,4,0.763501,2026-11-03 06:14:04.779640,2026-11-03 06:14:05.395377,0 days 00:00:00.615737,NaN,NaN,GradientBoosting,NaN,NaN,0.027136,12.0,8.0,10.0,64.0,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.789572,2026-11-03 06:14:30.186925,2026-11-03 06:14:30.210106,0 days 00:00:00.023181,NaN,0.147720,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifer"": ""SVM""}",COMPLETE
96,96,0.759777,2026-11-03 06:14:30.210733,2026-11-03 06:14:30.244803,0 days 00:00:00.034070,NaN,0.247060,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,"{""classifer"": ""SVM""}",COMPLETE
97,97,0.789572,2026-11-03 06:14:30.245471,2026-11-03 06:14:30.268513,0 days 00:00:00.023042,NaN,0.118536,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifer"": ""SVM""}",COMPLETE
98,98,0.787709,2026-11-03 06:14:30.269229,2026-11-03 06:14:30.297915,0 days 00:00:00.028686,NaN,0.179987,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifer"": ""SVM""}",COMPLETE


In [45]:
study.trials_dataframe()['params_classifer'].value_counts()

,count
params_classifer,
SVM,77
GradientBoosting,13
RandomForest,10


In [46]:
study.trials_dataframe().groupby('params_classifer')['value'].mean()

,value
params_classifer,
GradientBoosting,0.756339
RandomForest,0.763687
SVM,0.776464


In [47]:
# 1. Optimization History
plot_optimization_history(study).show()

In [48]:
plot_parallel_coordinate(study).show()

In [49]:
plot_slice(study).show()

In [51]:
plot_param_importances(study).show()